# Première analyse descriptive du fichier CSV
Ce notebook donne une vue d'ensemble du fichier : dimensions, colonnes, valeurs manquantes, statistiques, doublons et graphiques simples.

**À faire :** placez `consommation-annuelle-par-iris.csv` dans le même dossier que ce notebook, puis exécutez la cellule ci-dessous. Le code suppose un CSV séparé par une virgule ou un point-virgule.

In [3]:

# ============================================================
# ANALYSE DESCRIPTIVE DU FICHIER DE CONSOMMATION PAR IRIS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1. CHARGEMENT DU FICHIER
# Recherche le CSV dans le dossier courant ou dans le dossier Data.

chemins_possibles = [
    Path("consommation-annuelle-par-iris.csv"),
    Path("Data/consommation-annuelle-par-iris.csv")
]

fichier = next((p for p in chemins_possibles if p.exists()), None)

if fichier is None:
    raise FileNotFoundError(
        f"Fichier introuvable. Dossier courant : {Path.cwd()}"
    )

print("Fichier utilisé :", fichier.resolve())

# Détection simple du séparateur : virgule ou point-virgule
with open(fichier, "r", encoding="utf-8-sig") as f:
    extrait = f.read(10000)

separateur = ";" if extrait.count(";") > extrait.count(",") else ","

# Lecture du fichier CSV
df = pd.read_csv(
    fichier,
    sep=separateur,
    encoding="utf-8-sig",
    low_memory=False
)

print(f"\nFichier chargé : {df.shape[0]:,} lignes et {df.shape[1]} colonnes")


# 2. VUE D'ENSEMBLE
# Comprendre la taille du fichier et découvrir les variables.

print("\n" + "=" * 50)
print("1. APERÇU GÉNÉRAL")
print("=" * 50)

print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])
print("Séparateur utilisé :", repr(separateur))

print("\nPremières observations :")
display(df.head(10))

print("\nNoms des colonnes :")
print(df.columns.tolist())


# 3. TYPES ET QUALITÉ DES VARIABLES
# Repérer les types, les valeurs distinctes et les données manquantes.

print("\n" + "=" * 50)
print("2. TYPES ET VALEURS MANQUANTES")
print("=" * 50)

resume = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "valeurs_distinctes": df.nunique(dropna=True),
    "valeurs_manquantes": df.isna().sum(),
    "pourcentage_manquant": (df.isna().mean() * 100).round(2)
})

display(resume.sort_values(
    "pourcentage_manquant",
    ascending=False
))

print("\nNombre de lignes entièrement dupliquées :",
      df.duplicated().sum())


# 4. STATISTIQUES DESCRIPTIVES
# Résumer les variables numériques avec moyenne, médiane,
# écart-type, minimum, maximum et quartiles.

print("\n" + "=" * 50)
print("3. STATISTIQUES DES VARIABLES NUMÉRIQUES")
print("=" * 50)

colonnes_num = df.select_dtypes(include="number").columns

if len(colonnes_num) > 0:
    display(df[colonnes_num].describe().T)
else:
    print("Aucune variable numérique détectée.")

# Afficher les modalités les plus fréquentes des variables textuelles.
print("\n" + "=" * 50)
print("4. VARIABLES TEXTUELLES")
print("=" * 50)

colonnes_cat = df.select_dtypes(
    include=["object", "category", "string"]
).columns

for col in colonnes_cat:
    print(f"\nVariable : {col}")
    print("Nombre de modalités :", df[col].nunique(dropna=True))
    print(df[col].value_counts(dropna=False).head(5))


# 5. CONTRÔLE DES VALEURS NUMÉRIQUES
# Les valeurs négatives ne sont pas forcément des erreurs :
# elles doivent être interprétées selon la variable concernée.

print("\n" + "=" * 50)
print("5. VALEURS NUMÉRIQUES NÉGATIVES")
print("=" * 50)

if len(colonnes_num) > 0:
    display((df[colonnes_num] < 0).sum().to_frame(
        "nombre_de_valeurs_negatives"
    ))


# 6. GRAPHIQUES
# Tracer les distributions des six premières variables numériques
# pour repérer les asymétries et les valeurs extrêmes.

print("\n" + "=" * 50)
print("6. DISTRIBUTIONS DES VARIABLES NUMÉRIQUES")
print("=" * 50)

for col in colonnes_num[:6]:
    valeurs = df[col].dropna()

    if len(valeurs) == 0:
        continue

    plt.figure(figsize=(8, 4))
    plt.hist(valeurs, bins=30, edgecolor="black")
    plt.title(f"Distribution de {col}")
    plt.xlabel(col)
    plt.ylabel("Nombre d'observations")
    plt.tight_layout()
    plt.show()


# 7. REPÉRAGE DES VARIABLES TEMPORELLES ET GÉOGRAPHIQUES
# Rechercher dans les noms de colonnes des termes liés aux
# années, aux IRIS ou aux communes.

print("\n" + "=" * 50)
print("7. VARIABLES TEMPORELLES ET GÉOGRAPHIQUES")
print("=" * 50)

mots_cles = [
    "annee", "année", "year",
    "iris", "commune", "insee"
]

candidates = [
    col for col in df.columns
    if any(mot in col.lower() for mot in mots_cles)
]

if candidates:
    for col in candidates:
        exemples = df[col].dropna().astype(str).unique()[:8]
        print(f"\n{col}")
        print("Exemples de valeurs :", exemples.tolist())
else:
    print("Aucune colonne reconnue automatiquement.")


# 8. FIN DE L'ANALYSE

print("\n" + "=" * 50)
print("ANALYSE DESCRIPTIVE TERMINÉE")
print("=" * 50)

print(
    "Ces résultats permettent de découvrir la structure, "
    "la qualité et les principales distributions des données."
)
print(
    "Avant toute conclusion, vérifiez le sens des variables "
    "et leurs unités."
)


Fichier utilisé : C:\Users\dufau\Documents\GitHub\Projet_Impact_IA_Generative\Data\consommation-annuelle-par-iris.csv


ParserError: Error tokenizing data. C error: Expected 202 fields in line 3, saw 594
